In [9]:
from pathlib import Path

import pandas as pd

from amazonas_pipeline.constants import ISO3_TO_NAME

In [2]:
data_path = Path("./data")

In [ ]:
WANTED_COUNTRIES = [
    "Argentina",
    "Bahamas",
    "Barbados",
    "Belice",
    "Bolivia",
    "Brasil",
    "Chile",
    "Colombia",
    "Costa Rica",
    "Ecuador",
    "El Salvador",
    "Guatemala",
    "Guyana",
    "Haití",
    "Honduras",
    "Jamaica",
    "México",
    "Nicaragua",
    "Panamá",
    "Paraguay",
    "Perú",
    "República Dominicana",
    "Surinam",
    "Trinidad y Tobago",
    "Uruguay",
    "Venezuela",
]

country_to_iso3 = {v: k for k, v in ISO3_TO_NAME.items()}
wanted_isos = [country_to_iso3[country] for country in WANTED_COUNTRIES]

In [33]:
df_orig = pd.read_excel(
    data_path / "GHS-COUNTRY-STATS_MT_GLOBE_R2024_V1_0.xlsx",
    sheet_name=2,
).assign(
    country=lambda df: df["GADM_ISO"].map(ISO3_TO_NAME),
    DEGURBA_L1=lambda df: df["DEGURBA_L1"].map(
        {"RUR": "rural", "UC": "urban_center", "UCL": "urban_cluster"},
    ),
)

with pd.ExcelWriter("./CFS_populations.xlsx") as writer:
    for year in range(1975, 2021, 5):
        df = (
            df_orig.loc[lambda df: df["GADM_ISO"].isin(wanted_isos)]
            .pivot_table(index="country", columns="DEGURBA_L1", values=str(year))[
                ["rural", "urban_cluster", "urban_center"]
            ]
            .assign(total=lambda df: df.sum(axis=1))
        )
        df.to_excel(writer, sheet_name=str(year))

In [31]:
df

DEGURBA_L1,rural,urban_cluster,urban_center
country,,,
Argentina,5.110143e+06,6.840885e+06,1.392515e+07
Bahamas,6.275192e+04,6.779704e+04,7.133204e+04
Barbados,2.928496e+04,9.674875e+04,1.210563e+05
Belice,7.220336e+04,5.801110e+04,0.000000e+00
Bolivia,2.629485e+06,1.151496e+06,1.365361e+06
Brasil,3.370732e+07,3.875898e+07,3.621356e+07
Chile,3.290841e+06,2.370440e+06,4.978284e+06
Colombia,6.547000e+06,5.385951e+06,1.147679e+07
Costa Rica,8.427300e+05,5.638363e+05,7.068743e+05
